# 02 — Add PK, UNIQUE & FK Constraints

Declares informational constraints on our Delta tables and verifies them
via `information_schema.table_constraints`.

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "pk_unique_demo")

## Add Constraints

In [2]:
# Step 1: PK columns must be NOT NULL before adding the PRIMARY KEY constraint
spark.sql(f"ALTER TABLE {UC_CATALOG}.{UC_SCHEMA}.customers ALTER COLUMN customer_id SET NOT NULL")
spark.sql(f"ALTER TABLE {UC_CATALOG}.{UC_SCHEMA}.orders ALTER COLUMN order_id SET NOT NULL")
print("Set PK columns to NOT NULL")

# Step 2: Drop existing constraints (allows re-running this notebook safely)
# Order matters: drop FK first (references parent), then child PK, then parent constraints
drop_constraints = [
    (f"{UC_CATALOG}.{UC_SCHEMA}.orders", "orders_customer_fk"),
    (f"{UC_CATALOG}.{UC_SCHEMA}.orders", "orders_pk"),
    (f"{UC_CATALOG}.{UC_SCHEMA}.customers", "customers_email_unique"),
    (f"{UC_CATALOG}.{UC_SCHEMA}.customers", "customers_pk"),
]
for table, constraint_name in drop_constraints:
    try:
        spark.sql(f"ALTER TABLE {table} DROP CONSTRAINT {constraint_name}")
        print(f"  Dropped: {constraint_name}")
    except Exception as e:
        err = str(e).lower()
        if "not exist" in err or "cannot find" in err or "not found" in err or "no_such_constraint" in err:
            pass  # Constraint didn't exist — OK on first run
        else:
            raise
print("Ready to add constraints")

Set PK columns to NOT NULL
  Dropped: orders_customer_fk
  Dropped: orders_pk
  Dropped: customers_pk
Ready to add constraints


In [3]:
# Step 3: Add PK, UNIQUE, and FK constraints
# These are informational only (not enforced) — useful for optimizer and documentation
constraints = [
    f"ALTER TABLE {UC_CATALOG}.{UC_SCHEMA}.customers ADD CONSTRAINT customers_pk PRIMARY KEY (customer_id)",
    f"ALTER TABLE {UC_CATALOG}.{UC_SCHEMA}.customers ADD CONSTRAINT customers_email_unique UNIQUE (email)",
    f"ALTER TABLE {UC_CATALOG}.{UC_SCHEMA}.orders ADD CONSTRAINT orders_pk PRIMARY KEY (order_id)",
    f"ALTER TABLE {UC_CATALOG}.{UC_SCHEMA}.orders ADD CONSTRAINT orders_customer_fk FOREIGN KEY (customer_id) REFERENCES {UC_CATALOG}.{UC_SCHEMA}.customers(customer_id)",
]

for stmt in constraints:
    constraint_name = stmt.split("ADD CONSTRAINT ")[1].split(" ")[0]
    try:
        spark.sql(stmt)
        print(f"  Added: {constraint_name}")
    except Exception as e:
        if "UNIQUE_CONSTRAINT_DISABLED" in str(e):
            print(f"  Skipped: {constraint_name} (UNIQUE constraints not enabled on this workspace)")
            print(f"    To enable: SET spark.databricks.sql.dsv2.unique.enabled = true")
        else:
            raise

  Added: customers_pk


{"ts": "2026-03-23 15:39:20.589", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "\n[UNIQUE_CONSTRAINT_DISABLED] Unique constraint feature is disabled. To enable it, set \"spark.databricks.sql.dsv2.unique.enabled\" as true. SQLSTATE: 0A000\n== SQL (line 1, position 58) ==\n...th.pk_unique_demo.customers ADD CONSTRAINT customers_email_unique UNIQUE (email)\n                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n\n\nJVM stacktrace:\norg.apache.spark.sql.catalyst.parser.ParseException\n\tat org.apache.spark.sql.errors.QueryParsingErrors$.dsv2UniqueConstraintDisabledError(QueryParsingErrors.scala:880)\n\tat org.apache.spark.sql.catalyst.parser.AstBuilder.$anonfun$processDsv2Constraint$1(AstBuilder.scala:8333)\n\tat org.apache.spark.sql.catalyst.util.SparkParserUtils.withOrigin(SparkParserUtils.scala:213)\n\tat org.apache.spark.sql.catalyst.util.SparkParserUtils.withOrigin$(SparkParserUtils.scala:196)\n\tat org.apache.spark.sql.catalyst.parser.Pars

  Skipped: customers_email_unique (UNIQUE constraints not enabled on this workspace)
    To enable: SET spark.databricks.sql.dsv2.unique.enabled = true
  Added: orders_pk
  Added: orders_customer_fk


## Verify Constraints in information_schema

In [4]:
spark.sql(f"""
    SELECT constraint_name, constraint_type, table_name, enforced
    FROM {UC_CATALOG}.information_schema.table_constraints
    WHERE constraint_schema = '{UC_SCHEMA}'
    ORDER BY table_name, constraint_type
""").show(truncate=False)

+------------------+---------------+----------+--------+
|constraint_name   |constraint_type|table_name|enforced|
+------------------+---------------+----------+--------+
|customers_pk      |PRIMARY KEY    |customers |NO      |
|orders_customer_fk|FOREIGN KEY    |orders    |NO      |
|orders_pk         |PRIMARY KEY    |orders    |NO      |
+------------------+---------------+----------+--------+



Notice the `enforced` column — these constraints are **NOT enforced** by the engine.
They exist purely as metadata for the query optimizer and for documentation.

In the next notebook, we'll prove this by inserting data that violates every one of them.